<a href="https://colab.research.google.com/github/econ105/AI/blob/main/FINT501/AIagent/Buildingwikipediaagenttoolbox.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install --quiet wikipedia==1.4.0 langchain-core==0.3.59 pydantic==2.11.9

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.7/437.7 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.9/444.9 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 78.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 1.4.0 requires langchain-core<2.0.0,>=1.6.0, but you have langchain-core 0.3.59 which is incompatible.
google-adk 2.7.1 requires pydantic<3,>=2.12, but you have pydantic 2.11.9 which is incompatible.
google-genai 2.12.1

In [3]:
from typing import Annotated
import wikipedia
from langchain_core.tools import tool

wikipedia.set_user_agent(
    "company-research-tool/1.0 (https://www.datacamp.com; content@datacamp.com)"
)

@tool
def wikipedia_search(
    query: Annotated[str, "The Wikipedia search to execute to find key summary information."],
):
    """Use this to search Wikipedia for factual information."""
    try:
        # Step 1: Search using query
        results = wikipedia.search(query)

        if not results:
            return "No results found on Wikipedia."

        # Step 2: Retrieve page title
        title = results[0]

        # Step 3: Fetch summary
        summary = wikipedia.summary(title, sentences=8, auto_suggest=False, redirect=True)
    except BaseException as e:
        return f"Failed to execute. Error: {repr(e)}"
    return f"Successfully executed:\nWikipedia summary: {summary}"

In [4]:
import wikipedia

# Test with Apple query
company_name = "Apple Inc."
wiki_summary = wikipedia.summary(company_name)
print(wiki_summary)

Apple Inc. is an American multinational technology company headquartered in Cupertino, California, in Silicon Valley, and known for consumer electronics, software and online services. Founded in 1976 as Apple Computer Company by Steve Jobs, Steve Wozniak and Ronald Wayne, the company was incorporated by Jobs and Wozniak as Apple Computer, Inc. the following year. Its current name was adopted in 2007 as the company expanded its focus from computers to consumer electronics. Apple is one of the Big Tech companies.
The company was founded to market Wozniak's Apple I computer. Its successor, the Apple II, became one of the first successful mass-produced personal microcomputers. Apple introduced the Lisa in 1983 and the Macintosh in 1984, popularizing graphical user interfaces navigated by a mouse. By 1985, internal conflicts led to Jobs leaving the company to form NeXT and Wozniak stepping back from active employment. During the 1990s, Apple lost considerable market share to lower-priced Wi

Building your agent's toolbox
As you've seen, agents use tools to trigger certain actions. That could be executing some code, retrieving a local file, running a web search, or performing a math calculation.

As you progress through this course, you'll build a multi-agent system that can compile financial information, including summaries, stock performance data, and accompanying visualizations of Fortune 500 companies.

Let's start with a single-agent implementation to refresh ourselves on the LangGraph syntax and to create the tools we'll need for the multi-agent case.

In total, there are three tools required:

A tool for retrieving company information from the internet → Wikipedia
A tool for retrieving stock performance information from a local CSV file
A tool for running Python code to generate the visualizations
Let's start building these tools for your agents to use, beginning with a Wikipedia search tool!
![wikiimage](...)

🔎 Tool 1: Wikipedia search
You'll use the wikipedia Python library to perform Wikipedia searches and return informative summaries about different companies. This tool calls Wikipedia's API in the backend, which doesn't require an API key.

This library has two functions that we'll use to search and summarize company information:

wikipedia.search(): Search Wikipedia to return the highest-ranking page titles.
wikipedia.summary(): Fetch a summary from a given page. The sentences argument can be used to control the amount of information (number of sentences) returned.